# 02 — End-to-End UDA-Hub Demo

Runs the full LangGraph workflow on several sample tickets and shows:

1. Classification + routing decisions
2. Knowledge retrieval with confidence scoring
3. Tool invocation against the SQLite-backed support database
4. Resolution vs. escalation outcomes
5. Short-term (per-thread) and long-term (per-user) memory

> Requires `OPENAI_API_KEY` to be set in your environment (or `.env`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from uda_hub import db, seed
from uda_hub.config import settings
from uda_hub.retrieval import build_or_load_vectorstore
from uda_hub.graph import build_app
from uda_hub.logging_utils import configure_logging, get_run_log

configure_logging()
seed.seed_all(reset=True)              # fresh DB
build_or_load_vectorstore(rebuild=True)  # build FAISS index from KB

## 1. Compile the graph

`build_app` returns a compiled LangGraph app wired with a SQLite checkpointer (short-term memory)
and a SQLite long-term store. The Mermaid diagram below shows the agent topology.

In [ ]:
app = build_app()
print(app.get_graph().draw_mermaid())

## 2. Helper to run a single ticket

In [ ]:
import json, uuid
from uda_hub.runner import run_ticket

def demo(ticket: dict):
    print(f"\n{'='*72}\nTICKET {ticket['ticket_id']}: {ticket['subject']}\n{'='*72}")
    result = run_ticket(app, ticket)
    print('-- final answer --')
    print(result['answer'])
    print('-- routing trail --')
    for step in result['log']:
        print(' ', step)
    return result

## 3. Scenario A — Knowledge-base resolution

A simple how-to question that should be answered straight from the FAQ.

In [ ]:
demo({
    'ticket_id': 'tkt_demo_kb',
    'user_id':   'usr_001',
    'subject':   'How do I turn on 2FA?',
    'body':      "I want to enable two-factor authentication on my account.",
    'channel':   'web',
    'urgency':   'normal',
})

## 4. Scenario B — Tool-driven resolution (refund)

Customer with a billing issue. The Resolver should call the `process_refund` tool
and confirm the action.

In [ ]:
demo({
    'ticket_id': 'tkt_demo_refund',
    'user_id':   'usr_002',
    'subject':   'Charged twice again this month',
    'body':      'You billed me $19.99 twice on May 3rd. Please refund the duplicate.',
    'channel':   'email',
    'urgency':   'high',
})

## 5. Scenario C — Escalation (no relevant article)

Out-of-scope question with no matching article and a tool not safe to auto-execute.
Confidence should fall below threshold and the Escalation agent should take over.

In [ ]:
demo({
    'ticket_id': 'tkt_demo_esc',
    'user_id':   'usr_003',
    'subject':   'Bulk export still failing — clients impacted',
    'body':      "Our nightly bulk export to S3 returns 500. Three of our enterprise clients are blocked.",
    'channel':   'email',
    'urgency':   'critical',
})

## 6. Scenario D — Memory in action

Same user, follow-up question in the same thread. The graph should pick up the
checkpointed state (short-term memory) and remember the prior context.

In [ ]:
demo({
    'ticket_id': 'tkt_demo_followup',
    'user_id':   'usr_002',
    'subject':   'Follow-up to my refund',
    'body':      "Quick check — when should I see the refund on my statement?",
    'channel':   'email',
    'urgency':   'normal',
    'thread_id': 'usr_002-billing',  # same thread as scenario B
})

## 7. Inspect long-term memory for a returning customer

In [ ]:
from uda_hub.memory import get_long_term_store
store = get_long_term_store()
for item in store.search(('customer', 'usr_002')):
    print(item.key, '->', item.value)

## 8. Inspect persisted ticket state in SQLite

In [ ]:
from tabulate import tabulate
print(tabulate(db.fetch_all("""
    SELECT t.ticket_id, t.status, m.category, m.urgency, m.sentiment, m.confidence, m.routed_to
      FROM Ticket t JOIN TicketMetadata m USING(ticket_id)
     WHERE t.ticket_id LIKE 'tkt_demo_%'
     ORDER BY t.ticket_id
"""), headers='keys'))

In [ ]:
print(tabulate(db.fetch_all("""
    SELECT ticket_id, role, author, substr(content,1,80) AS preview
      FROM TicketMessage
     WHERE ticket_id LIKE 'tkt_demo_%'
     ORDER BY message_id
"""), headers='keys'))

---

Every decision (classification, routing, retrieval, tool call, escalation) is logged via
`uda_hub.logging_utils` and written to `data/uda_hub.db` via `TicketMessage`/`TicketMetadata`
for full audit replay.